### Setup

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from pathlib import Path
import json
import random

random.seed(42)

### Task 1: Connect via SQLAlchemy and load the full 'restaurants' table with pd.read_sql()

In [ ]:
# Create a local SQLite database standing in for the MySQL/PostgreSQL Zomato backend
engine = create_engine("sqlite:///zomato_backend.db")

restaurants_df = pd.DataFrame({
    "restaurant_id": range(1, 11),
    "name": [f"Restaurant_{i}" for i in range(1, 11)],
    "cuisine": ["North Indian", "Chinese", "Italian", "South Indian", "Continental",
                "Fast Food", "North Indian", "Italian", "Chinese", "South Indian"],
    "rating": [4.2, 3.8, 4.5, 4.0, 3.5, 4.1, 3.9, 4.6, 3.7, 4.3],
    "city": ["Mumbai", "Delhi", "Bangalore", "Surat", "Pune",
             "Hyderabad", "Mumbai", "Delhi", "Bangalore", "Surat"]
})
restaurants_df.to_sql("restaurants", engine, if_exists="replace", index=False)

In [ ]:
# Load the entire 'restaurants' table using pd.read_sql()
restaurants_from_db = pd.read_sql("restaurants", engine)

print("First 5 rows of the restaurants table:")
restaurants_from_db.head(5)

### Task 2: Use pd.read_sql_query() to fetch only 'name' and 'rating' where rating > 8

In [ ]:
# Create a 'movies' table (BookMyShow-style data) in the same database
movies_df = pd.DataFrame({
    "movie_id": range(1, 11),
    "name": [f"Movie_{i}" for i in range(1, 11)],
    "genre": ["Action", "Drama", "Comedy", "Thriller", "Romance",
              "Action", "Drama", "Comedy", "Thriller", "Romance"],
    "rating": [8.5, 7.2, 9.1, 6.8, 8.9, 7.5, 8.2, 6.5, 9.3, 7.8]
})
movies_df.to_sql("movies", engine, if_exists="replace", index=False)

# Custom SQL SELECT query as the first argument to pd.read_sql_query()
query = "SELECT name, rating FROM movies WHERE rating > 8"
high_rated_movies = pd.read_sql_query(query, engine)

print("Movies with rating above 8:")
print(high_rated_movies)

### Task 3: Read a JSON dataset directly from a URL

In [ ]:
# Read directly from the URL - pandas can fetch and parse JSON straight from an endpoint
users_url = "https://jsonplaceholder.typicode.com/users"
users_df = pd.read_json(users_url)

print("Usernames from the API:")
print(users_df["username"])

### Task 4: Load orders.csv and users.csv using pathlib, then merge on 'user_id'

In [ ]:
# Create the two sample CSV files
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

users_data = pd.DataFrame({
    "user_id": range(1, 6),
    "username": ["asha", "rohit", "priya", "karan", "meera"]
})
users_data.to_csv(data_dir / "users.csv", index=False)

orders_data = pd.DataFrame({
    "order_id": range(101, 111),
    "user_id": [1, 2, 1, 3, 4, 5, 2, 3, 4, 1],
    "amount": [250, 480, 150, 620, 300, 900, 210, 540, 175, 330]
})
orders_data.to_csv(data_dir / "orders.csv", index=False)

In [ ]:
# Load both files using pathlib for the file paths
orders_path = Path("data") / "orders.csv"
users_path = Path("data") / "users.csv"

orders_df = pd.read_csv(orders_path)
users_df = pd.read_csv(users_path)

# Merge on 'user_id' to show a combined table with username and amount
combined_df = orders_df.merge(users_df, on="user_id")
print("Combined orders + usernames:")
combined_df[["order_id", "user_id", "username", "amount"]]

### Task 5: Concatenate 'today_orders' and 'yesterday_orders' with pd.concat()

In [ ]:
today_orders = pd.DataFrame({
    "order_id": [201, 202, 203],
    "item": ["Pizza", "Burger", "Pasta"],
    "price": [350, 180, 420]
})

yesterday_orders = pd.DataFrame({
    "order_id": [198, 199, 200],
    "item": ["Sushi", "Salad", "Sandwich"],
    "price": [500, 220, 150]
})

# Concatenate and reset the index after concatenation
combined_orders = pd.concat([today_orders, yesterday_orders]).reset_index(drop=True)

print("Combined orders (today + yesterday):")
combined_orders